# 📈 Regression Analysis — D1–D5

Phân tích mối liên hệ giữa kích thước code, thời gian đến lần sửa và các đặc trưng của AI commit. Đây là phân tích liên hệ, không tự chứng minh quan hệ nhân quả.

Cần cài thêm: `statsmodels`, `patsy`.

In [2]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from patsy import dmatrices
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
DATA_PATH = Path('../data/processed/commits_features.csv')
df = pd.read_csv(DATA_PATH, low_memory=False)
df['days_to_modification'] = pd.to_numeric(df['days_to_modification'], errors='coerce')
df['is_modified'] = df['is_modified'].astype(bool)
df = df.replace([np.inf, -np.inf], np.nan)
REPORT_DIR = Path('../reports'); FIG_DIR = REPORT_DIR / 'figures'; FIG_DIR.mkdir(parents=True, exist_ok=True)
def savefig(name):
    plt.tight_layout(); plt.savefig(FIG_DIR / name, dpi=160, bbox_inches='tight'); plt.show(); plt.close()
print(f'Đã tải {len(df):,} dòng × {df.shape[1]} cột')

Đã tải 33,917 dòng × 24 cột


## D1 — Hồi quy tuyến tính đơn
`code_lines ~ days_to_modification`

In [3]:
d1 = df[['code_lines','days_to_modification']].dropna()
d1_model = smf.ols('code_lines ~ days_to_modification', data=d1).fit(cov_type='HC3')
print(d1_model.summary())
d1_result = pd.DataFrame({'term': d1_model.params.index, 'coef': d1_model.params.values, 'p_value': d1_model.pvalues.values, 'ci_lower': d1_model.conf_int()[0].values, 'ci_upper': d1_model.conf_int()[1].values})
d1_result.to_csv(REPORT_DIR / 'regression_d1_simple.csv', index=False)
plt.figure(figsize=(8,5)); sns.regplot(data=d1.sample(min(10000,len(d1)), random_state=42), x='days_to_modification', y='code_lines', scatter_kws={'alpha':.25,'s':15}, line_kws={'color':'red'}); plt.title('D1: code_lines theo days_to_modification'); savefig('D1_simple_regression.png')

                            OLS Regression Results                            
Dep. Variable:             code_lines   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     69.55
Date:                Tue, 20 Oct 2026   Prob (F-statistic):           8.39e-17
Time:                        20:54:33   Log-Likelihood:                -70859.
No. Observations:               10475   AIC:                         1.417e+05
Df Residuals:                   10473   BIC:                         1.417e+05
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept              116.9037 

## D2 — Hồi quy tuyến tính bội
`days_to_modification ~ code_lines + repo_commit_count + language + ai_tool + task_type`

In [4]:
d2_cols = ['days_to_modification','code_lines','repo_commit_count','language','ai_tool','task_type']
d2 = df[d2_cols].dropna().copy()
d2_model = smf.ols('days_to_modification ~ code_lines + repo_commit_count + C(language) + C(ai_tool) + C(task_type)', data=d2).fit(cov_type='HC3')
print(d2_model.summary())
d2_result = pd.DataFrame({'term': d2_model.params.index, 'coef': d2_model.params.values, 'p_value': d2_model.pvalues.values, 'ci_lower': d2_model.conf_int()[0].values, 'ci_upper': d2_model.conf_int()[1].values})
d2_result.to_csv(REPORT_DIR / 'regression_d2_multiple.csv', index=False)
plt.figure(figsize=(9, max(5, len(d2_result)*.25))); coef = d2_result[d2_result.term != 'Intercept'].sort_values('coef'); plt.errorbar(coef.coef, range(len(coef)), xerr=[coef.coef-coef.ci_lower, coef.ci_upper-coef.coef], fmt='o'); plt.yticks(range(len(coef)), coef.term); plt.axvline(0,color='black',lw=1); plt.title('D2: coefficient plot'); plt.xlabel('Hệ số (95% CI)'); savefig('D2_coefficient_plot.png')

                             OLS Regression Results                             
Dep. Variable:     days_to_modification   R-squared:                       0.028
Model:                              OLS   Adj. R-squared:                  0.027
Method:                   Least Squares   F-statistic:                     29.07
Date:                  Tue, 20 Oct 2026   Prob (F-statistic):           8.80e-87
Time:                          20:54:34   Log-Likelihood:                -58545.
No. Observations:                 10475   AIC:                         1.171e+05
Df Residuals:                     10458   BIC:                         1.172e+05
Df Model:                            16                                         
Covariance Type:                    HC3                                         
                                             coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------

## D3 — Hồi quy logistic
`is_modified ~ code_lines + repo_commit_count + language + ai_tool + task_type + year`

In [5]:
d3_cols = ['is_modified','code_lines','repo_commit_count','language','ai_tool','task_type','year']
d3 = df[d3_cols].dropna().copy()
d3['is_modified'] = d3['is_modified'].astype(int)
d3_model = smf.glm('is_modified ~ code_lines + repo_commit_count + C(language) + C(ai_tool) + C(task_type) + year', data=d3, family=sm.families.Binomial()).fit(cov_type='HC3')
print(d3_model.summary())
d3_result = pd.DataFrame({'term': d3_model.params.index, 'log_odds_coef': d3_model.params.values, 'odds_ratio': np.exp(d3_model.params.values), 'p_value': d3_model.pvalues.values, 'ci_lower_or': np.exp(d3_model.conf_int()[0].values), 'ci_upper_or': np.exp(d3_model.conf_int()[1].values)})
d3_result.to_csv(REPORT_DIR / 'regression_d3_logistic.csv', index=False)
plt.figure(figsize=(9, max(5, len(d3_result)*.25))); coef = d3_result[d3_result.term != 'Intercept'].sort_values('odds_ratio'); plt.errorbar(coef.odds_ratio, range(len(coef)), xerr=[coef.odds_ratio-coef.ci_lower_or, coef.ci_upper_or-coef.odds_ratio], fmt='o'); plt.yticks(range(len(coef)), coef.term); plt.axvline(1,color='black',lw=1); plt.xscale('log'); plt.title('D3: odds ratio coefficient plot'); plt.xlabel('Odds ratio (95% CI)'); savefig('D3_logistic_coefficients.png')

                 Generalized Linear Model Regression Results                  
Dep. Variable:            is_modified   No. Observations:                33917
Model:                            GLM   Df Residuals:                    33899
Model Family:                Binomial   Df Model:                           17
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -21391.
Date:                Tue, 20 Oct 2026   Deviance:                       42782.
Time:                        20:54:34   Pearson chi2:                 3.39e+04
No. Iterations:                     5   Pseudo R-squ. (CS):            0.05618
Covariance Type:                  HC3                                         
                                             coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------

## D4 — R², p-value và VIF

In [6]:
model_summary = pd.DataFrame([
    {'model':'D1 OLS', 'n':int(d1_model.nobs), 'r_squared':d1_model.rsquared, 'adj_r_squared':d1_model.rsquared_adj, 'aic':d1_model.aic},
    {'model':'D2 OLS multiple', 'n':int(d2_model.nobs), 'r_squared':d2_model.rsquared, 'adj_r_squared':d2_model.rsquared_adj, 'aic':d2_model.aic},
    {'model':'D3 GLM logistic', 'n':int(d3_model.nobs), 'r_squared':np.nan, 'adj_r_squared':np.nan, 'aic':d3_model.aic}
])
display(model_summary); model_summary.to_csv(REPORT_DIR / 'regression_model_summary.csv', index=False)
_, X = dmatrices('days_to_modification ~ code_lines + repo_commit_count + C(language) + C(ai_tool) + C(task_type)', data=d2, return_type='dataframe')
vif = pd.DataFrame({'term': X.columns, 'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]})
display(vif); vif.to_csv(REPORT_DIR / 'regression_d2_vif.csv', index=False)

,model,n,r_squared,adj_r_squared,aic
0,D1 OLS,10475,0.002376,0.002281,141721.067903
1,D2 OLS multiple,10475,0.028166,0.026679,117124.950715
2,D3 GLM logistic,33917,NaN,NaN,42817.534361


,term,VIF
0,Intercept,1.000000
1,C(language)[T.python],1.120934
2,C(ai_tool)[T.claude],2.151856
3,C(ai_tool)[T.cursor],1.028258
4,C(ai_tool)[T.deepseek],1.009337
5,C(ai_tool)[T.gemini_bard],1.148979
6,C(ai_tool)[T.github_copilot],1.282846
7,C(ai_tool)[T.other_ai],1.382547
8,C(ai_tool)[T.unknown],1.305211
9,C(task_type)[T.data_query],1.223275


## D5 — Residual plot
Residual plot kiểm tra xem sai số của D2 có hình dạng bất thường hay không.

In [7]:
pred = d2_model.get_prediction(d2).summary_frame()
plt.figure(figsize=(8,5)); sns.scatterplot(x=pred['mean'], y=d2_model.resid, s=12, alpha=.25); plt.axhline(0,color='red',ls='--'); plt.xlabel('Giá trị dự đoán'); plt.ylabel('Residual'); plt.title('D2 residual plot'); savefig('D5_residual_plot.png')
print('Các file kết quả đã lưu trong reports/ và reports/figures/.')

Các file kết quả đã lưu trong reports/ và reports/figures/.
